In [ ]:
#### LIESTAL WITH COMBIPRECIP DATA 

In [32]:
import sys
sys.path.append("/storage/homefs/ge24z347/LISFLOOD_FP_8_1/src")
import importlib
import rasterio
import DEM_processing  # Import your updated module
import lisflood_inputdata
import flow_depth_plotting # Import your updated module

importlib.reload(DEM_processing)  # Reload the updated module
importlib.reload(lisflood_inputdata)
importlib.reload(flow_depth_plotting)  # Reload the updated module
import geopandas as gpd
import pandas as pd

In [18]:
import xarray as xr

# Replace with the path to your NetCDF file
# Open the dataset
ds_Liestal = xr.open_dataset('/storage/homefs/ge24z347/LISFLOOD_FP_8_1/Data_forprocess/Combiprecip/CPC_00060_H_20240624000000_20240630230000.nc')
ds_Liestal

<xarray.Dataset> Size: 305MB
Dimensions:       (REFERENCE_TS: 168, x: 710, y: 640)
Coordinates:
  * REFERENCE_TS  (REFERENCE_TS) datetime64[ns] 1kB 2024-06-24 ... 2024-06-30...
  * x             (x) float64 6kB 2.256e+06 2.256e+06 ... 2.964e+06 2.964e+06
  * y             (y) float64 5kB 1.48e+06 1.478e+06 ... 8.415e+05 8.405e+05
Data variables:
    CPC           (REFERENCE_TS, y, x) float32 305MB ...
Attributes:
    CDI:              Climate Data Interface version 1.9.9 (https://mpimet.mp...
    Conventions:      CF-1.8
    Source_Software:  FME
    history:          Wed Aug 21 07:40:32 2024: cdo mergetime /scratch2/mkauz...
    CDO:              Climate Data Operators version 1.9.9 (https://mpimet.mp...

In [21]:
# 2. Define your selected REFERENCE_TS times
selected_times = np.array([
    '2024-06-25T15:00:00',
    '2024-06-25T16:00:00',
    '2024-06-25T17:00:00',
    '2024-06-25T18:00:00',
    '2024-06-25T19:00:00',
    '2024-06-25T20:00:00',
    '2024-06-25T21:00:00',
    '2024-06-25T22:00:00',
    '2024-06-25T23:00:00',
    '2024-06-26T00:00:00',
    '2024-06-26T01:00:00'
], dtype='datetime64[ns]')

# 3. DEM bounding box (from your message)
left, right = 2619000, 2623000
bottom, top = 1257000, 1261000

# 4. Crop both time and spatial domain
ds_crop = ds_Liestal.sel(
    REFERENCE_TS=selected_times,
    x=slice(left, right),
    y=slice(top, bottom)  # y is usually descending
)

ds_crop

<xarray.Dataset> Size: 856B
Dimensions:       (REFERENCE_TS: 11, x: 4, y: 4)
Coordinates:
  * REFERENCE_TS  (REFERENCE_TS) datetime64[ns] 88B 2024-06-25T15:00:00 ... 2...
  * x             (x) float64 32B 2.62e+06 2.62e+06 2.622e+06 2.622e+06
  * y             (y) float64 32B 1.26e+06 1.26e+06 1.258e+06 1.258e+06
Data variables:
    CPC           (REFERENCE_TS, y, x) float32 704B ...
Attributes:
    CDI:              Climate Data Interface version 1.9.9 (https://mpimet.mp...
    Conventions:      CF-1.8
    Source_Software:  FME
    history:          Wed Aug 21 07:40:32 2024: cdo mergetime /scratch2/mkauz...
    CDO:              Climate Data Operators version 1.9.9 (https://mpimet.mp...

In [16]:
#!/usr/bin/env python3
import os
import numpy as np
import xarray as xr
import rioxarray as rxr
from netCDF4 import Dataset
from datetime import date

def crop_deterministic_Combiprecip(
    orig_nc: str,
    dem_file: str,
    output_nc: str,
    selected_times: list
):
    """
    Crop deterministic Combiprecip file to DEM footprint and specific REFERENCE_TS times.
    Save output as a single NetCDF with dims (time, y, x) and variable rainfall_depth.
    """

    # 1) Read DEM bounds
    dem = (
        rxr.open_rasterio(dem_file, masked=True)
           .sel(band=1)
           .rio.write_crs("EPSG:2056")
    )
    left, bottom, right, top = dem.rio.bounds()
    print(f"DEM bounds ▶ X {left:.0f}→{right:.0f}, Y {bottom:.0f}→{top:.0f}")

    # 2) Open NetCDF and select REFERENCE_TS and spatial subset
    ds = xr.open_dataset(orig_nc)

    # Ensure selected_times are in numpy datetime64 format
    selected_times = np.array(selected_times, dtype='datetime64[ns]')

    # Select by time and spatial bounding box (NOTE: y goes from top to bottom!)
    ds_sel = (
        ds.sel(REFERENCE_TS=selected_times)
          .sel(x=slice(left, right), y=slice(top, bottom))  # 🔁 FIXED HERE
    )
    print(f"Selected {len(ds_sel.REFERENCE_TS)} time steps and cropped spatially.")

    # 3) Access CPC variable and clean up
    da = ds_sel['CPC'].transpose("REFERENCE_TS", "y", "x")
    data = np.nan_to_num(da.values.astype(np.float32), nan=0.0)
    t = da.REFERENCE_TS.values.astype('datetime64[ns]')
    x = da.x.values.astype(np.float32)
    y = da.y.values.astype(np.float32)
    nt, ny, nx = data.shape

    # 4) Write to output NetCDF
    nc = Dataset(output_nc, "w")

    # dimensions
    nc.createDimension("time", nt)
    nc.createDimension("x",    nx)
    nc.createDimension("y",    ny)

    # coords
    tv = nc.createVariable("time", "f8", ("time",))
    xv = nc.createVariable("x",    "f4", ("x",))
    yv = nc.createVariable("y",    "f4", ("y",))

    tv.units = "seconds since 1970-01-01 00:00:00"
    tv.axis = "T"
    xv.units = "m"
    xv.axis = "X"
    yv.units = "m"
    yv.axis = "Y"

    tv[:] = (t.astype('datetime64[s]').astype(float))
    xv[:] = x
    yv[:] = y

    # data variable
    rv = nc.createVariable(
        "rainfall_depth", "f4",
        ("time", "y", "x"),
        zlib=True, complevel=4, shuffle=True
    )
    rv.units = "mm"
    rv.standard_name = "precipitation_amount"
    rv[:] = data

    # global attributes
    nc.description = "Cropped deterministic CPC rainfall"
    nc.history     = f"Created on {date.today().isoformat()}"
    nc.source      = "CPC deterministic forecast cropped to DEM footprint"

    nc.close()
    print(f"✔ Saved: {output_nc}")


if __name__ == "__main__":
    crop_deterministic_Combiprecip(
        orig_nc = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/Data_forprocess/Combiprecip/CPC_00060_H_20240624000000_20240630230000.nc",
        dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Liestal_2m/Liestal_2m_DEM.tif",
        output_nc = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Liestal_2m/Liestal_2m.nc",
        selected_times = [
            '2024-06-25T15:00:00',
            '2024-06-25T16:00:00',
            '2024-06-25T17:00:00',
            '2024-06-25T18:00:00',
            '2024-06-25T19:00:00',
            '2024-06-25T20:00:00',
            '2024-06-25T21:00:00',
            '2024-06-25T22:00:00',
            '2024-06-25T23:00:00',
            '2024-06-26T00:00:00',
            '2024-06-26T01:00:00'
        ]
    )

/storage/homefs/ge24z347/mambaforge/envs/env_py311/lib/python3.11/site-packages/pyproj/crs/_cf1x8.py:515: UserWarning: angle from rectified to skew grid parameter lost in conversion to CF
  warnings.warn(


DEM bounds ▶ X 2619000→2623000, Y 1257000→1261000
Selected 11 time steps and cropped spatially.
✔ Saved: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Liestal_2m/Liestal_2m.nc


In [5]:
from DEM_processing import create_par_file_Liestal_Combiprecip

create_par_file_Liestal_Combiprecip(
    base_name = "Liestal_2m",
    output_file_path = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Liestal_2m/Liestal_2m.par"
)

✔ File created at: /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Liestal_2m/Liestal_2m.par


In [10]:
from DEM_processing import create_stage_file 

create_stage_file(
    catchment_location_csv="/storage/homefs/ge24z347/LISFLOOD_FP_8_1/Data_forprocess/catchment_location.csv",
    selected_id=2,
    output_stage_file="/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Liestal_2m/Liestal_2m.stage"
)

.stage file created successfully at /storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Liestal_2m/Liestal_2m.stage


In [ ]:
## Plotting the 10 hours of the combiprecip 

In [33]:
### Plotting Liestal event 15 UTC until 5 hours lead time
from flow_depth_plotting import g_plots_selected_wd_liestal_dinamic_no_cbar

g_plots_selected_wd_liestal_dinamic_no_cbar(dem_file = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Liestal_2m/Liestal_2m.dem",
wd_folder = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/build/Liestal_2m/Liestal_2m/",
plot_output_folder = "/storage/homefs/ge24z347/LISFLOOD_FP_8_1/LIESTAL_PLOTS/Liestal_2m_Combiprecip/",
geo_ezgg_2km_ge = "/rs_scratch/users/ge24z347/geo_ezgg_2km_ge.shp",
plot_title_prefix = "Liestal",
initial_datetime_str="2024-06-25T15:00:00",
lead_times_hours=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
color1="navajowhite", color2="darkorange", color3="firebrick",
xlim=(2.62075e6, 2.6225e6), ylim=(1.25875e6, 1.25975e6))

ValueError: Could not extract ensemble number from folder name: Liestal_2m